# YOLO ?????????Kaggle GPU ??
**??**
1. ?? `yolo_detect_dataset_kaggle.zip` ??? Kaggle Dataset????? Create ? New Dataset??????????? Add Input ? Upload?
2. ?? Settings ? Accelerator ? **GPU T4 x2**????
3. ? Run All???? 100 ??GPU ??? 1~3 ??
4. ????? `/kaggle/working/best.pt`????? `E:\apple\yolo_detect_best.pt`?????????

**??**
- ???? 8246 ????? apple?train/val ????
- ????????? data.yaml ????????????
- ???? GPU ???????? CPU??????????


In [ ]:
!pip install -q ultralytics

In [ ]:
import os, glob

# ????????/kaggle/input ?? data.yaml ????
input_dir = "/kaggle/input"
dataset_root = None
for root, dirs, files in os.walk(input_dir):
    if "data.yaml" in files:
        dataset_root = root
        break
if dataset_root is None:
    raise FileNotFoundError("???????????? Add Input ?????????")
print("?????:", dataset_root)
os.system("ls -la " + dataset_root)

# ?? data.yaml????? Windows ????????path:?
yaml_path = os.path.join(dataset_root, "data.yaml")
with open(yaml_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
keep = [ln for ln in lines if not ln.strip().lower().startswith("path:")]
if len(keep) != len(lines):
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.writelines(keep)
    print("??? data.yaml??????????????")
    with open(yaml_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("data.yaml ????")


In [ ]:
import torch, time
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
is_cpu = device == "cpu"
print("????:", device, "(GPU??)" if not is_cpu else "(CPU?????????GPU)")

model = YOLO("yolo11n.pt")  # ??????????

# GPU?100 ? + 640 ??? + ?????CPU??????
t0 = time.time()
model.train(
    data=os.path.join(dataset_root, "data.yaml"),
    epochs=20 if is_cpu else 100,
    imgsz=320 if is_cpu else 640,
    batch=8 if is_cpu else 16,
    device=device,
    project="/kaggle/working/yolo_detect_output",
    name="apple_detect", exist_ok=True,
    patience=10 if is_cpu else 30,
    augment=True,
    workers=4, cos_lr=True, warmup_epochs=3,
    seed=0,
)
sec = time.time() - t0
print(f"??????? {sec/60:.1f} ??")


In [ ]:
import os, shutil

best = "/kaggle/working/yolo_detect_output/apple_detect/weights/best.pt"
if os.path.exists(best):
    shutil.copy2(best, "/kaggle/working/best.pt")
    size_mb = os.path.getsize(best) / 1e6
    print(f"?????: /kaggle/working/best.pt ({size_mb:.1f} MB)")
    print("???????????? E:\\apple\\yolo_detect_best.pt")
else:
    print("best.pt ????????????")
